# BƯỚC 3: PHÂN MẢNH NGỮ CẢNH (CHUNKING) & LƯU TRỮ TRUNG GIAN
Sử dụng tokenizer của model Embedding để cắt nhỏ văn bản, bọc metadata và xuất ra file `.json` vào thư mục `json_chunking` để lưu trữ và kiểm tra trước khi đẩy vào Vector DB.

In [1]:
import os
import json
import unicodedata
from datetime import datetime, timezone
from transformers import AutoTokenizer
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

def normalize_vietnamese_text(text: str) -> str:
    """Chuẩn hóa Unicode tiếng Việt về chuẩn dựng sẵn NFC"""
    return unicodedata.normalize("NFC", text)

/home/trung-ai/miniconda3/envs/ocr_docs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_markdown_for_rag_production(markdown_text: str, tokenizer, source_path: str):
    """Tiến hành phân mảnh văn bản dựa trên cấu trúc Markdown linh hoạt."""
    clean_text = normalize_vietnamese_text(markdown_text)

    # 1. Định nghĩa các Header tiềm năng - Tự động bắt mọi cấp độ tiêu đề
    headers_to_split_on = [
        ("#", "H1"),
        ("##", "H2"),
        ("###", "H3"),
        ("####", "H4"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on,
        strip_headers=False
    )
    md_header_splits = markdown_splitter.split_text(clean_text)
    
    # 2. Sử dụng Splitter thông minh bảo vệ Bảng (Table)
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=tokenizer,
        chunk_size=250,
        chunk_overlap=40,
        separators=[
            "\n\n",           # Đoạn văn
            "</table>",         # ƯU TIÊN 1: Kết thúc bảng
            "\n## ", "\n### ", "\n####" # ƯU TIÊN 2: Bắt đầu Điều/Mục mới
            "\n",             # Dòng
            ". ", "? ", "! ",
            " ", ""
        ]
    )
    
    final_docs = []
    source_name = os.path.basename(source_path)
    ingested_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    for header_doc in md_header_splits:
        sub_chunks = text_splitter.split_documents([header_doc])
        
        # --- LOGIC BREADCRUMB LINH HOẠT ---
        header_metadata = header_doc.metadata
        sorted_keys = sorted(header_metadata.keys()) # Đảm bảo thứ tự H1 > H2 > H3
        breadcrumb_parts = [header_metadata[k] for k in sorted_keys]
        breadcrumb = " > ".join(breadcrumb_parts)

        for doc in sub_chunks:
            # Ghi đè page_content với context (breadcrumb)
            if breadcrumb:
                doc.page_content = f"NGỮ CẢNH: {breadcrumb}\n\n{doc.page_content}"

            # Metadata chuẩn hóa
            doc.metadata = {
                "source":       source_name,
                "breadcrumb":   breadcrumb,
                "char_count":   len(doc.page_content),
                "ingested_at":  ingested_at,
            }
            final_docs.append(doc)

    # Đánh chỉ mục chunk
    for idx, doc in enumerate(final_docs):
        doc.metadata["chunk_index"] = idx

    return [c for c in final_docs if c.metadata["char_count"] > 100]

In [3]:
def export_chunks_to_json(chunks, output_dir: str, original_filename: str):
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(original_filename))[0]
    json_path = os.path.join(output_dir, f"{base_name}_chunks.json")
    
    data_to_save = [{"page_content": c.page_content, "metadata": c.metadata} for c in chunks]
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data_to_save, f, ensure_ascii=False, indent=4)
    return json_path

In [4]:
from pathlib import Path

# Cấu hình đường dẫn
model_id_or_path = "/home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m" 
input_dir = "/home/trung-ai/chatbot_hcns/create_documents/md_final_files"
staging_dir = "/home/trung-ai/chatbot_hcns/create_documents/json_chunking"


print(f"⏳ Đang load Tokenizer từ model local: {model_id_or_path}...")
tokenizer = AutoTokenizer.from_pretrained(model_id_or_path)

⏳ Đang load Tokenizer từ model local: /home/trung-ai/chatbot_hcns/weights/embeddinggemma-300m...


In [5]:
import glob

md_files = glob.glob(os.path.join(input_dir, "*.md"))

if not md_files:
    print(f"❌ Không tìm thấy file .md nào trong thư mục: {input_dir}")
else:
    print(f"🚀 Tìm thấy {len(md_files)} file markdown. Bắt đầu quá trình chunking...")
    
    total_chunks = 0
    for file_path in md_files:
        file_name = os.path.basename(file_path)
        print(f"\n--- ⏳ Đang xử lý file: {file_name} ---")
        
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                markdown_text = f.read()
            
            chunks = process_markdown_for_rag_production(
                markdown_text, tokenizer,
                source_path=file_path,
            )
            
            saved_json_path = export_chunks_to_json(chunks, staging_dir, file_path)
            
            total_chunks += len(chunks)
            print(f"✅ Thành công: {len(chunks)} chunks -> {os.path.basename(saved_json_path)}")
            
        except Exception as e:
            print(f"❌ Lỗi khi xử lý file {file_name}: {e}")

    print("\n" + "=" * 60)
    print(f"✅ HOÀN TẤT TẤT CẢ FILE!")
    print(f"📊 Tổng số file đã xử lý: {len(md_files)}")
    print(f"📦 Tổng số chunks đã tạo: {total_chunks}")
    print(f"📁 Dữ liệu JSON được lưu tại: {staging_dir}")
    print("=" * 60)

🚀 Tìm thấy 17 file markdown. Bắt đầu quá trình chunking...

--- ⏳ Đang xử lý file: Quy định nghỉ việc_cleaned.md ---
✅ Thành công: 29 chunks -> Quy định nghỉ việc_cleaned_chunks.json

--- ⏳ Đang xử lý file: HR-AIPT.BỘ QUY TẮC ỨNG XỬ 2025_cleaned.md ---
✅ Thành công: 88 chunks -> HR-AIPT.BỘ QUY TẮC ỨNG XỬ 2025_cleaned_chunks.json

--- ⏳ Đang xử lý file: Sổ tay chấm công_cleaned.md ---
✅ Thành công: 6 chunks -> Sổ tay chấm công_cleaned_chunks.json

--- ⏳ Đang xử lý file: 250820_HR_QĐ_11_Quy_định_về_Cấp_phát_và_Sử_dụng_Đồng_phục_cleaned.md ---
✅ Thành công: 12 chunks -> 250820_HR_QĐ_11_Quy_định_về_Cấp_phát_và_Sử_dụng_Đồng_phục_cleaned_chunks.json

--- ⏳ Đang xử lý file: E_QUY_TRÌNH_HDSD_QUY_TRÌNH_ĐỀ_XUẤT_LÀM_THÊM_GIỜ_final_cleaned.md ---
✅ Thành công: 14 chunks -> E_QUY_TRÌNH_HDSD_QUY_TRÌNH_ĐỀ_XUẤT_LÀM_THÊM_GIỜ_final_cleaned_chunks.json

--- ⏳ Đang xử lý file: HR-AIPT.QUYẾT ĐỊNH.QUY ĐỊNH LÀM THÊM GIỜ_cleaned.md ---
✅ Thành công: 11 chunks -> HR-AIPT.QUYẾT ĐỊNH.QUY ĐỊNH LÀM THÊM GIỜ_cleane